In [1]:
from google.colab import files
uploaded = files.upload()

Saving test_set.csv to test_set.csv


In [2]:
import pandas as pd

df = pd.read_csv("test_set.csv")

print(df.head())
print(df.columns)
print("Number of rows:", len(df))

               id                                               text  \
0  hatexplain_684               i am half black and i grew up muslim   
1   jigsaw_133950       No, it really isn't. I'll remove it now... -   
2    jigsaw_61896  (Thanks, N.h., for catching the header that I ...   
3    jigsaw_25578  "\nThis is not a reason to speedily delete it....   
4   jigsaw_114641             == Sorry == \n\n I am sorry Sarah...:(   

   text_length source_dataset abusive  
0           36     HateXplain     NOT  
1           44         JIGSAW     NOT  
2           54         JIGSAW     NOT  
3          225         JIGSAW     NOT  
4           36         JIGSAW     NOT  
Index(['id', 'text', 'text_length', 'source_dataset', 'abusive'], dtype='object')
Number of rows: 22777


In [3]:
import pandas as pd
import random
import re
import unicodedata

# Makes experiments reproducible
RANDOM_SEED = 42

random.seed(RANDOM_SEED)

Character Substitution Generator

In [4]:
CHAR_SUBSTITUTIONS = {
    "a": ["4", "@"],
    "b": ["8"],
    "e": ["3"],
    "g": ["9"],
    "i": ["1", "!"],
    "l": ["1"],
    "o": ["0"],
    "s": ["5", "$"],
    "t": ["7"],
    "z": ["2"]
}


def character_substitution(text, severity=0.3):
    """
    Replace characters with visually similar numbers/symbols.

    severity:
        0.0 = no changes
        1.0 = replace every eligible character
    """

    output = []

    for char in str(text):

        lower_char = char.lower()

        if (
            lower_char in CHAR_SUBSTITUTIONS
            and random.random() < severity
        ):
            replacement = random.choice(
                CHAR_SUBSTITUTIONS[lower_char]
            )

            output.append(replacement)

        else:
            output.append(char)

    return "".join(output)

Spacing Manipulation Generator

In [5]:
def spacing_manipulation(text, severity=0.3):
    """
    Insert spaces inside words.

    Higher severity results in more inserted spaces.
    """

    output = []

    for char in str(text):

        output.append(char)

        # Only insert spaces after letters
        if (
            char.isalpha()
            and random.random() < severity
        ):
            output.append(" ")

    result = "".join(output)

    # Remove excessive spaces
    result = re.sub(r" {2,}", " ", result)

    return result.strip()

Symbolic Masking

In [6]:
MASKING_SYMBOLS = [
    "*",
    "#",
    "_",
    ".",
    "-"
]


def symbolic_masking(text, severity=0.3):
    """
    Replace selected alphabetic characters
    with masking symbols.
    """

    output = []

    for char in str(text):

        if (
            char.isalpha()
            and random.random() < severity
        ):
            output.append(
                random.choice(MASKING_SYMBOLS)
            )

        else:
            output.append(char)

    return "".join(output)

Misspelling Generator

In [7]:
ALPHABET = "abcdefghijklmnopqrstuvwxyz"


def misspell_word(word, severity=0.3):

    if len(word) < 3:
        return word

    if random.random() > severity:
        return word

    operation = random.choice([
        "delete",
        "swap",
        "duplicate",
        "replace"
    ])

    position = random.randint(
        1,
        len(word) - 2
    )

    # Delete character
    if operation == "delete":

        return (
            word[:position]
            + word[position + 1:]
        )

    # Swap two neighbouring characters
    elif operation == "swap":

        chars = list(word)

        chars[position], chars[position + 1] = (
            chars[position + 1],
            chars[position]
        )

        return "".join(chars)

    # Duplicate character
    elif operation == "duplicate":

        return (
            word[:position]
            + word[position]
            + word[position:]
        )

    # Replace character
    elif operation == "replace":

        replacement = random.choice(ALPHABET)

        return (
            word[:position]
            + replacement
            + word[position + 1:]
        )


def misspelling(text, severity=0.3):

    words = str(text).split()

    transformed = [
        misspell_word(word, severity)
        for word in words
    ]

    return " ".join(transformed)

Unicode Confusable Substitution

In [8]:
UNICODE_CONFUSABLES = {
    "a": ["а"],       # Cyrillic a
    "c": ["с"],       # Cyrillic es
    "e": ["е"],       # Cyrillic ie
    "i": ["і"],       # Cyrillic i
    "j": ["ј"],       # Cyrillic je
    "o": ["о"],       # Cyrillic o
    "p": ["р"],       # Cyrillic er
    "s": ["ѕ"],       # Cyrillic dze
    "x": ["х"],       # Cyrillic ha
    "y": ["у"]        # Cyrillic u
}


def unicode_confusable(text, severity=0.3):
    """
    Replace Latin characters with visually
    similar Unicode characters.
    """

    output = []

    for char in str(text):

        lower_char = char.lower()

        if (
            lower_char in UNICODE_CONFUSABLES
            and random.random() < severity
        ):

            replacement = random.choice(
                UNICODE_CONFUSABLES[lower_char]
            )

            output.append(replacement)

        else:
            output.append(char)

    return "".join(output)

Combination of Techniques

In [9]:
OBFUSCATION_FUNCTIONS = {
    "character_substitution": character_substitution,
    "spacing": spacing_manipulation,
    "symbolic_masking": symbolic_masking,
    "misspelling": misspelling,
    "unicode_confusable": unicode_confusable
}


def obfuscate_text(
    text,
    technique,
    severity=0.3
):

    if technique not in OBFUSCATION_FUNCTIONS:
        raise ValueError(
            f"Unknown technique: {technique}"
        )

    function = OBFUSCATION_FUNCTIONS[
        technique
    ]

    return function(
        text,
        severity=severity
    )

Severity Levels

In [10]:
SEVERITY_LEVELS = {
    "low": 0.15,
    "medium": 0.35,
    "high": 0.60
}

Single-Technique Attack Dataset

In [11]:
def generate_attack_dataset(df):

    results = []

    for _, row in df.iterrows():

        original_text = str(
            row["text"]
        )

        # Preserve clean sample
        clean_row = row.to_dict()

        clean_row["original_text"] = (
            original_text
        )

        clean_row["obfuscated_text"] = (
            original_text
        )

        clean_row["obfuscation_type"] = (
            "clean"
        )

        clean_row["severity"] = (
            "none"
        )

        results.append(clean_row)

        # Generate attacks
        for technique in OBFUSCATION_FUNCTIONS:

            for severity_name, severity_value in SEVERITY_LEVELS.items():

                new_row = row.to_dict()

                new_row["original_text"] = (
                    original_text
                )

                new_row["obfuscated_text"] = (
                    obfuscate_text(
                        original_text,
                        technique,
                        severity_value
                    )
                )

                new_row["obfuscation_type"] = (
                    technique
                )

                new_row["severity"] = (
                    severity_name
                )

                results.append(
                    new_row
                )

    return pd.DataFrame(results)

In [12]:
attack_df = generate_attack_dataset(df)

attack_df.head(20)

,id,text,text_length,source_dataset,abusive,original_text,obfuscated_text,obfuscation_type,severity
0,hatexplain_684,i am half black and i grew up muslim,36,HateXplain,NOT,i am half black and i grew up muslim,i am half black and i grew up muslim,clean,none
1,hatexplain_684,i am half black and i grew up muslim,36,HateXplain,NOT,i am half black and i grew up muslim,i @m ha1f bl@ck 4nd i gr3w up muslim,character_substitution,low
2,hatexplain_684,i am half black and i grew up muslim,36,HateXplain,NOT,i am half black and i grew up muslim,i 4m ha1f 81@ck @nd i grew up muslim,character_substitution,medium
3,hatexplain_684,i am half black and i grew up muslim,36,HateXplain,NOT,i am half black and i grew up muslim,! 4m h4lf black @nd i 9r3w up musl1m,character_substitution,high
4,hatexplain_684,i am half black and i grew up muslim,36,HateXplain,NOT,i am half black and i grew up muslim,i am half bla ck an d i grew up m uslim,spacing,low
5,hatexplain_684,i am half black and i grew up muslim,36,HateXplain,NOT,i am half black and i grew up muslim,i am half b lack an d i gr ew up m u s lim,spacing,medium
6,hatexplain_684,i am half black and i grew up muslim,36,HateXplain,NOT,i am half black and i grew up muslim,i a m hal f b l ack a n d i gre w u p musl i m,spacing,high
7,hatexplain_684,i am half black and i grew up muslim,36,HateXplain,NOT,i am half black and i grew up muslim,i am half black and * gre* up m#slim,symbolic_masking,low
8,hatexplain_684,i am half black and i grew up muslim,36,HateXplain,NOT,i am half black and i grew up muslim,i a. h*lf bl*_k #.# # g--w up m.sli*,symbolic_masking,medium
9,hatexplain_684,i am half black and i grew up muslim,36,HateXplain,NOT,i am half black and i grew up muslim,i .- h#*- _---# *n* i ---_ u_ ._*-im,symbolic_masking,high


Multi-Obfuscation Attack

In [13]:
def multi_obfuscation(
    text,
    techniques=None,
    severity=0.3
):

    if techniques is None:

        techniques = random.sample(
            list(
                OBFUSCATION_FUNCTIONS.keys()
            ),
            2
        )

    transformed = text

    for technique in techniques:

        transformed = (
            OBFUSCATION_FUNCTIONS[
                technique
            ](
                transformed,
                severity
            )
        )

    return (
        transformed,
        techniques
    )

Multi-Obfuscation Dataset

In [14]:
def generate_multi_attack_dataset(df):

    results = []

    for _, row in df.iterrows():

        original_text = str(
            row["text"]
        )

        for severity_name, severity_value in SEVERITY_LEVELS.items():

            new_row = row.to_dict()

            transformed, techniques = (
                multi_obfuscation(
                    original_text,
                    severity=severity_value
                )
            )

            new_row["original_text"] = (
                original_text
            )

            new_row["obfuscated_text"] = (
                transformed
            )

            new_row["obfuscation_type"] = (
                "+".join(techniques)
            )

            new_row["attack_count"] = (
                len(techniques)
            )

            new_row["severity"] = (
                severity_name
            )

            results.append(
                new_row
            )

    return pd.DataFrame(results)

In [15]:
multi_attack_df = (
    generate_multi_attack_dataset(df)
)

multi_attack_df.head()

,id,text,text_length,source_dataset,abusive,original_text,obfuscated_text,obfuscation_type,attack_count,severity
0,hatexplain_684,i am half black and i grew up muslim,36,HateXplain,NOT,i am half black and i grew up muslim,. .m half black and i gre. up muol_m,misspelling+symbolic_masking,2,low
1,hatexplain_684,i am half black and i grew up muslim,36,HateXplain,NOT,i am half black and i grew up muslim,і аm half black @nd і 9rew uр muslim,unicode_confusable+character_substitution,2,medium
2,hatexplain_684,i am half black and i grew up muslim,36,HateXplain,NOT,i am half black and i grew up muslim,* -. _*af b*nc# ad * #r#w up mussl#.,misspelling+symbolic_masking,2,high
3,jigsaw_133950,"No, it really isn't. I'll remove it now... -",44,JIGSAW,NOT,"No, it really isn't. I'll remove it now... -","No, it rea1ly isn't. I'll remove it now... -",character_substitution+misspelling,2,low
4,jigsaw_133950,"No, it really isn't. I'll remove it now... -",44,JIGSAW,NOT,"No, it really isn't. I'll remove it now... -","N, it really іsn't. I'll reove it now... -",unicode_confusable+misspelling,2,medium


Applying onto Abusive Text ONLY

In [16]:
abusive_df = df[
    df["abusive"] == "OFF"
].copy()

print(
    "Abusive samples:",
    len(abusive_df)
)

Abusive samples: 2723


In [17]:
attack_df = (
    generate_attack_dataset(
        abusive_df
    )
)

In [18]:
multi_attack_df = (
    generate_multi_attack_dataset(
        abusive_df
    )
)

<= 280 Characters after Obfuscation

In [19]:
attack_df["obfuscated_length"] = (
    attack_df[
        "obfuscated_text"
    ]
    .astype(str)
    .str.len()
)

attack_df = attack_df[
    attack_df[
        "obfuscated_length"
    ] <= 280
].copy()

In [20]:
multi_attack_df["obfuscated_length"] = (
    multi_attack_df[
        "obfuscated_text"
    ]
    .astype(str)
    .str.len()
)

multi_attack_df = multi_attack_df[
    multi_attack_df[
        "obfuscated_length"
    ] <= 280
].copy()

Download

In [21]:
attack_df.to_csv(
    "single_obfuscation_attacks.csv",
    index=False
)

multi_attack_df.to_csv(
    "multi_obfuscation_attacks.csv",
    index=False
)

In [22]:
from google.colab import files

files.download(
    "single_obfuscation_attacks.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
files.download(
    "multi_obfuscation_attacks.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>